In [17]:
import pandas as pd
import numpy as np
import random
import sklearn.manifold as manifold
import colorsys
import json
import re
import subprocess
import uuid
from pathlib import Path

import plotly.express as px
from IPython.display import HTML, display

In [18]:
# Read distance matrix
# dist_mat = pd.read_csv("../output/handshape_distance_matrix.csv")
dist_mat = pd.read_csv("../output/handshape_distance_matrix_handskel.csv")

dist_mat.head()

,Name,1_1.FBX,1_1_Jitter.FBX,1_2-2.FBX,1_2-3.FBX,1_2.FBX,1_Curved_Jitter-2.FBX,1_Curved_Jitter-3.FBX,1_Curved_Jitter.FBX,3.FBX,...,M20260506_0604_260713_0.fbx_8,M20260506_0604_260713_0.fbx_9,M20260506_0604_260713_0.fbx_10,M20260506_0604_260713_0.fbx_11,M20260506_0604_260713_0.fbx_12,M20260506_0604_260713_0.fbx_13,M20260506_0604_260713_0.fbx_14,M20260506_0604_260713_0.fbx_15,M20260506_0604_260713_0.fbx_16,M20260506_0604_260713_0.fbx_17
0,1_1.FBX,0.00000,1.29154,1.700850,1.596630,1.47584,2.35544,1.553600,1.62458,5.38570,...,6.04173,4.91861,5.71563,5.26966,4.48650,4.86900,5.71476,4.55041,4.59883,6.15265
1,1_1_Jitter.FBX,1.29154,0.00000,2.363280,2.217570,2.37235,3.15161,2.195690,2.33210,6.23857,...,6.56947,5.43428,6.27716,5.68757,4.97631,5.18502,5.48280,4.91136,5.04519,6.65184
2,1_2-2.FBX,1.70085,2.36328,0.000000,0.485771,1.26979,1.02582,0.501303,1.34755,5.48031,...,6.76501,5.82437,6.41839,6.24105,3.34458,5.90541,6.72117,5.58300,3.45103,6.91825
3,1_2-3.FBX,1.59663,2.21757,0.485771,0.000000,1.14081,1.30048,0.352566,1.21294,5.74841,...,6.67437,5.75588,6.32854,6.17867,3.49161,5.86530,6.57244,5.54104,3.60417,6.82653
4,1_2.FBX,1.47584,2.37235,1.269790,1.140810,0.00000,1.85650,1.278640,1.00781,5.18883,...,5.76250,4.84035,5.35977,5.33519,4.24326,5.11052,6.71320,4.82476,4.37325,5.91196


In [19]:
# Top 10 most similar handshapes to 'fbx\M20241107_6263_260710_2.fbx_segment_2'

# Get the row for the specified handshape
# Get file name from index 2
index = 69
file_name = dist_mat.iloc[index, 0]
handshape_row = dist_mat.iloc[index]

# Get the indices of the 10 smallest distances
handshape_distances = handshape_row[1:].to_numpy()
similar_indices = handshape_distances.argsort()

# Get the names of the 10 most similar handshapes (excluding any names starting with 'fbx\')
total = 0
similar_handshape_names = []
for index in similar_indices:
    if total >= 10:
        break
    handshape_name = dist_mat.columns[index + 1]  # +1 to account for the 'Name' column
    if handshape_name.endswith('.FBX'):
        similar_handshape_names.append(handshape_name)
        total += 1

print("10 most similar handshapes to " + file_name + ":")
for handshape in similar_handshape_names:
    print(handshape)

10 most similar handshapes to M20260119_1078_260713_1.fbx_6:
Beak2_Open.FBX
R.FBX
C2_Spread.FBX
L2.FBX
Beak2_Open_Spread.FBX
V_Curved.FBX
D.FBX
D-2.FBX
W.FBX
V.FBX


In [20]:
# Get distance array
dist_mat_clean = dist_mat.drop(columns=['Name'])
distance_array = dist_mat_clean.to_numpy()

In [21]:
# # Make all distances 0 or higher
# distance_array += np.abs(np.min(distance_array))

In [22]:
# t-SNE on the distance matrix
random_num = random.randint(0, 10000)
# random_num = 8079  # This is the last one I used, so for reproducibility hardcoded
print("Using random seed:", random_num)
tsne_dist = manifold.TSNE(n_components=2, perplexity=50, learning_rate=200, init='random', metric='precomputed', method='barnes_hut', random_state=random_num)
X_tsne_dist = tsne_dist.fit_transform(distance_array)

Using random seed: 6347


In [24]:
tsneDf = pd.DataFrame({
    'tsne_1': X_tsne_dist[:, 0],
    'tsne_2': X_tsne_dist[:, 1],
    'name': dist_mat_clean.columns,
    'is_handshape': [name.endswith(".FBX") for name in dist_mat_clean.columns]
})

projectRoot = next(
    root for root in (Path.cwd(), Path.cwd().parent)
    if (root / 'fbx').is_dir() and (root / 'python').is_dir()
)
helperPath = projectRoot / 'python' / 'fbx_hand_skeleton.exe'
animationPattern = re.compile(r'^(.*\.fbx)_(\d+)$', re.IGNORECASE)
animationFiles = sorted({
    match.group(1)
    for name in dist_mat_clean.columns
    if (match := animationPattern.match(name))
})

skeletonsByName = {}
for animationFile in animationFiles:
    result = subprocess.run(
        [helperPath, projectRoot / 'fbx' / animationFile],
        check=True,
        capture_output=True,
        text=True
    )
    skeletonsByName.update(json.loads(result.stdout))

handshapeNames = [name for name in dist_mat_clean.columns if name.endswith('.FBX')]
handshapeIndices = [dist_mat_clean.columns.get_loc(name) for name in handshapeNames]
closestHandshapeByName = {}
for animationName in skeletonsByName:
    animationIndex = dist_mat_clean.columns.get_loc(animationName)
    handshapeDistances = dist_mat_clean.iloc[animationIndex, handshapeIndices]
    closestHandshape = handshapeDistances.idxmin()
    closestHandshapeByName[animationName] = {
        'name': closestHandshape,
        'distance': float(handshapeDistances[closestHandshape])
    }

fig = px.scatter(
    tsneDf,
    x='tsne_1',
    y='tsne_2',
    color='is_handshape',
    hover_data={'name': True, 'is_handshape': False},
    custom_data=['name'],
    title="t-SNE Dimensionality Reduction of the Shape Features — click an animation point to inspect it"
)

fig.update_layout(width=800, height=800)
fig.update_traces(marker={'size': 8})

instanceId = uuid.uuid4().hex
plotId = f'tsne-plot-{instanceId}'
modalId = f'hand-modal-{instanceId}'
handPlotId = f'hand-plot-{instanceId}'
titleId = f'hand-title-{instanceId}'
closeId = f'hand-close-{instanceId}'

figHtml = fig.to_html(
    full_html=False,
    include_plotlyjs=True,
    div_id=plotId,
    config={'displaylogo': False}
)

modalHtml = f"""
<style>
#{modalId} {{
    display: none;
    width: 800px;
    max-width: 100%;
    margin-top: 12px;
}}
#{modalId} .hand-modal-content {{
    width: 100%;
    height: 720px;
    padding: 16px;
    border-radius: 10px;
    background: white;
    border: 1px solid #d0d0d0;
    box-sizing: border-box;
}}
#{titleId} {{
    display: inline-block;
    max-width: calc(100% - 52px);
    margin: 0 0 8px;
    overflow-wrap: anywhere;
    font: 600 16px sans-serif;
}}
#{closeId} {{
    float: right;
    border: 0;
    background: transparent;
    font-size: 28px;
    line-height: 24px;
    cursor: pointer;
}}
#{handPlotId} {{
    width: 100%;
    height: calc(100% - 36px);
}}
</style>
<div id="{modalId}">
    <div class="hand-modal-content">
        <div id="{titleId}"></div>
        <button id="{closeId}" type="button" aria-label="Close">&times;</button>
        <div id="{handPlotId}"></div>
    </div>
</div>
<script>
(() => {{
    const plot = document.getElementById({json.dumps(plotId)});
    const modal = document.getElementById({json.dumps(modalId)});
    const handPlot = document.getElementById({json.dumps(handPlotId)});
    const title = document.getElementById({json.dumps(titleId)});
    const closeButton = document.getElementById({json.dumps(closeId)});
    const skeletonsByName = {json.dumps(skeletonsByName)};
    const closestHandshapeByName = {json.dumps(closestHandshapeByName)};
    const fingerJoints = {{
        thumb: ['thumb_metacarpal', 'thumb_proximal', 'thumb_distal', 'thumb_tip'],
        index: ['index_metacarpal', 'index_proximal', 'index_middle', 'index_distal', 'index_tip'],
        middle: ['middle_metacarpal', 'middle_proximal', 'middle_middle', 'middle_distal', 'middle_tip'],
        ring: ['ring_metacarpal', 'ring_proximal', 'ring_middle', 'ring_distal', 'ring_tip'],
        pinky: ['pinky_metacarpal', 'pinky_proximal', 'pinky_middle', 'pinky_distal', 'pinky_tip']
    }};
    const fingerColors = {{
        thumb: '#e63946',
        index: '#f4a261',
        middle: '#2a9d8f',
        ring: '#457b9d',
        pinky: '#7b2cbf'
    }};

    const closeModal = () => {{
        modal.style.display = 'none';
    }};

    const isValidPoint = point => Array.isArray(point) && point.length === 3 && point.every(Number.isFinite);

    plot.on('plotly_click', event => {{
        const name = event.points[0].customdata[0];
        const skeleton = skeletonsByName[name];
        if (!skeleton) {{
            window.alert(`No hand skeleton data found for ${{name}}`);
            return;
        }}

        const traces = [];
        for (const [finger, jointNames] of Object.entries(fingerJoints)) {{
            const points = jointNames.map(jointName => skeleton[jointName]).filter(isValidPoint);
            traces.push({{
                type: 'scatter3d',
                mode: 'lines+markers',
                name: finger,
                x: points.map(point => point[0]),
                y: points.map(point => point[1]),
                z: points.map(point => point[2]),
                line: {{color: fingerColors[finger], width: 8}},
                marker: {{color: fingerColors[finger], size: 5}}
            }});
        }}

        const palmPoints = ['thumb_metacarpal', 'index_metacarpal', 'middle_metacarpal', 'ring_metacarpal', 'pinky_metacarpal']
            .map(jointName => skeleton[jointName])
            .filter(isValidPoint);
        traces.push({{
            type: 'scatter3d',
            mode: 'lines+markers',
            name: 'palm',
            x: palmPoints.map(point => point[0]),
            y: palmPoints.map(point => point[1]),
            z: palmPoints.map(point => point[2]),
            line: {{color: '#333333', width: 5}},
            marker: {{color: '#333333', size: 4}}
        }});

        const closestHandshape = closestHandshapeByName[name];
        title.textContent = closestHandshape
            ? `${{name}} — closest handshape: ${{closestHandshape.name}} (distance: ${{closestHandshape.distance.toFixed(4)}})`
            : name;
        modal.style.display = 'block';
        Plotly.react(handPlot, traces, {{
            margin: {{l: 0, r: 0, b: 0, t: 10}},
            legend: {{orientation: 'h'}},
            scene: {{
                aspectmode: 'data',
                xaxis: {{title: 'X'}},
                yaxis: {{title: 'Y'}},
                zaxis: {{title: 'Z'}}
            }}
        }}, {{displaylogo: false, responsive: true}});
    }});

    closeButton.addEventListener('click', closeModal);
    modal.addEventListener('click', event => {{
        if (event.target === modal) closeModal();
    }});
    document.addEventListener('keydown', event => {{
        if (event.key === 'Escape') closeModal();
    }});
}})();
</script>
"""

display(HTML(figHtml + modalHtml))

FileNotFoundError: [WinError 2] Het systeem kan het opgegeven bestand niet vinden